# PolySurrogate sample workflow

This small example runs the complete pipeline:

1. generate graph weights and exact quantum amplitudes;
2. train a polynomial neural-network (PNN) surrogate;
3. use the surrogate for inverse design.

The settings are intentionally small and demonstrate the software rather than reproduce the paper results.


## 1. Setup


In [ ]:
from pathlib import Path
import json

import jax
import numpy as np

import data_generate
import inverse_design
import ml_model

REPO_ROOT = Path.cwd().resolve()
if not all((REPO_ROOT / name).exists() for name in ("data_generate.py", "ml_model.py", "inverse_design.py")):
    raise RuntimeError("Launch this notebook from the PolySurrogate repository root.")

SEED = 42
NODES = 4
DIMENSIONS = 2
N_SAMPLES = 5_000
BATCH_SIZE = 200
NUM_EPOCHS = 20
HIDDEN_DIM = 400
OPT_STEPS = 300
RESULTS_ROOT = REPO_ROOT / "results" / "sample_workflow"

print(f"JAX backend: {jax.default_backend()}")
print(f"Outputs: {RESULTS_ROOT.relative_to(REPO_ROOT)}")


## 2. Generate training data

Each sample maps 24 graph-edge weights to a 16-component quantum amplitude vector. Production training uses much larger datasets; this example uses 5,000 samples for speed.


In [ ]:
weights, amplitudes, data_dir = data_generate.generate_dataset(
    vertices=NODES,
    dimensions=DIMENSIONS,
    n_samples=N_SAMPLES,
    batch_size=500,
    seed=SEED,
    shard_size=N_SAMPLES,
    normed_data=False,
    save_data=True,
    data_root=str(RESULTS_ROOT / "data_generation"),
)
dataset_path = data_dir / "dataset_merged.npz"
print(f"Dataset: {dataset_path.relative_to(REPO_ROOT)}")
print(f"Weights: {weights.shape}; amplitudes: {amplitudes.shape}")


## 3. Train the surrogate

The PNN learns the mapping from graph weights to amplitudes. The resulting `best_params.msgpack` file is used by inverse design.


In [ ]:
ml_model.NODES = NODES
ml_model.MODEL_NAME = "PNN"
ml_model.HIDDEN_DIM = HIDDEN_DIM
ml_model.DATA_PATH = str(dataset_path)
ml_model.DATA_SIZE = None
ml_model.SPLIT_MODE = "contiguous"
ml_model.BATCH_SIZE = BATCH_SIZE
ml_model.NUM_EPOCHS = NUM_EPOCHS
ml_model.PATIENCE = NUM_EPOCHS
ml_model.LR_DECAY_UNTIL_EPOCH = NUM_EPOCHS
ml_model.SEED = SEED
ml_model.ROOT_FOLDER = str(RESULTS_ROOT / "model_training")
ml_model.RESUME_RUN_DIR = None
ml_model.CONFIG.update(
    nodes=NODES,
    model_name="PNN",
    hidden_dim=HIDDEN_DIM,
    data_path=str(dataset_path),
    data_size=None,
    split_mode="contiguous",
    batch_size=BATCH_SIZE,
    num_epochs=NUM_EPOCHS,
    patience=NUM_EPOCHS,
    lr_decay_until_epoch=NUM_EPOCHS,
    seed=SEED,
)

model_dir = ml_model.train()
model_path = model_dir / "best_params.msgpack"
with (model_dir / "summary.json").open() as file:
    training_summary = json.load(file)

print(f"Model: {model_path.relative_to(REPO_ROOT)}")
print(f"Test MAE: {training_summary['test_mae']:.6f}")
print(f"Test fidelity: {training_summary['test_fidelity']:.6f}")


## 4. Run inverse design

One generated amplitude vector is used as a custom target. The optimiser searches for graph weights that reproduce it and verifies the final state with PyTheus.


In [ ]:
rng = np.random.default_rng(SEED)
sample_index = int(rng.integers(0, N_SAMPLES))
target_state = amplitudes[sample_index].astype(np.float32)

config = inverse_design._default_cfg()
config.update(
    n=NODES,
    dimensions=DIMENSIONS,
    target_name="CUSTOM",
    target_state=target_state.tolist(),
    model_type="PNN",
    generate_data=True,
    data_samples=1,
    data_batch_size=1,
    data_seed=SEED,
    generation_gpu_batch_size=1,
    data_shard_size=1,
    normed_data=False,
    architecture=HIDDEN_DIM,
    model_path=str(model_path),
    normalize_model_output=False,
    input_dim=2 * NODES * (NODES - 1),
    out_dim=2**NODES,
    seed=SEED,
    num_steps=OPT_STEPS,
    max_total_steps=OPT_STEPS,
    lr_decay_steps=OPT_STEPS,
    results_root=str(RESULTS_ROOT / "inverse_design"),
)
result_dir = inverse_design.run_optimisation(config)


In [ ]:
with (Path(result_dir) / "best_graph_solution.json").open() as file:
    result = json.load(file)

print(f"Target sample: {sample_index}")
print(f"PyTheus fidelity: {result['fidelity']:.4f}")
print(f"Active edges: {result['num_edges']} / {2 * NODES * (NODES - 1)}")
print(f"Results: {Path(result_dir).relative_to(REPO_ROOT)}")


## Next steps

The workflow is complete when all cells run without error. Increase the dataset size, training epochs, and optimisation steps for research runs. The main settings are documented in the repository README and near the top of each script.
